# 09 — Conversation and Long-Context Engineering

## Scenario
A customer starts a chat, mentions their order number, gets distracted by asking a bunch of unrelated questions about shipping policies, and then finally asks for an update on their order.

**The Problem:** We cannot pass the *entire* history of every user into every prompt. It is too expensive, too slow, and leads to "Lost in the Middle" hallucinations. We need a strategy to persist state.

In [ ]:
import os
import json
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

# Initialize the client (requires GEMINI_API_KEY environment variable)
client = genai.Client()
MODEL_ID = 'gemini-2.5-flash'

# The Chat History
chat_history = [
    {"role": "user", "content": "Hi, I need help with my order ORD-5592."},
    {"role": "assistant", "content": "I can help with that. What do you need to know?"},
    {"role": "user", "content": "Actually, before we get to that, do you ship to Alaska?"},
    {"role": "assistant", "content": "Yes, we ship to Alaska, but it takes an extra 2 days."},
    {"role": "user", "content": "What about Hawaii?"},
    {"role": "assistant", "content": "Yes, Hawaii also takes an extra 2 days."},
    # ... Imagine 10 more turns of unrelated conversation here ...
]

current_request = "Okay great. Anyway, can you give me an update on my order?"


## Strategy 1: The Sliding Window

The simplest approach is to only pass the last N messages to the LLM to save tokens.

In [ ]:
# We only keep the last 4 messages
sliding_window = chat_history[-4:]

context = "\n".join([f"{msg['role']}: {msg['content']}" for msg in sliding_window])
prompt = f"""You are a helpful assistant.\nRecent Chat History:\n{context}\n\nUser: {current_request}\n"""

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt
)
print("--- Sliding Window Output ---")
print(response.text)

# Notice: The model has completely forgotten the Order ID (ORD-5592) because it was truncated from the window. 
# It will have to ask the user to repeat themselves, creating a terrible user experience.

## Strategy 2: Summary Memory

Instead of truncating, we can ask an LLM to periodically summarize the conversation and pass the summary along with the sliding window.

In [ ]:
# Imagine a background process generated this summary of the truncated messages
chat_summary = "The user needs help with order ORD-5592. They also asked about shipping to Alaska and Hawaii."

prompt = f"""You are a helpful assistant.\nConversation Summary: {chat_summary}\nRecent Chat History:\n{context}\n\nUser: {current_request}\n"""

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt
)
print("--- Summary Memory Output ---")
print(response.text)

# Notice: The model succeeds because the Order ID was in the summary.
# The Danger: Summaries degrade. If the conversation goes on for 100 turns, the summary of summaries 
# often drops specific details like IDs to save space.

## Strategy 3: Structured State Extraction (State of the Art)

The gold standard for production AI (used by agent frameworks) is to maintain a strict, structured "State Object". After every user message, an LLM extracts critical facts into this schema.

In [ ]:
class UserState(BaseModel):
    active_order_id: str | None = Field(description="The active order ID if the user provided one, otherwise null.")
    user_intent: str = Field(description="What the user is trying to accomplish.")

# Imagine this state was extracted during turn 1 of the conversation and saved to a Postgres database.
current_state = UserState(active_order_id="ORD-5592", user_intent="Check order status")

prompt = f"""You are a helpful assistant.\nUser State:\n{current_state.model_dump_json(indent=2)}\n\nRecent Chat History:\n{context}\n\nUser: {current_request}\n"""

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt
)
print("--- Structured State Output ---")
print(response.text)

# Conclusion: By defining a strict contract for memory (UserState), we guarantee that critical 
# facts are never forgotten, dropped by a summarizer, or lost in a sliding window.